# Download and stage a 50k SynthHuman pilot on Colab

This notebook acquires only the first 50,000 SynthHuman samples (10 release ZIP chunks), retaining ZIP archives on Drive. Set RUN_50K_DOWNLOAD=True in the download cell when ready. The staging cell checks local Colab disk capacity before extracting the first 10 chunks under /content/SynthHuman_staged. It does not modify the already-extracted dataset in Drive.

In [ ]:
# Default off. Set True only when ready for the 50k pilot archive transfer.
RUN_50K_DOWNLOAD = False
SAMPLE_COUNT_TO_DOWNLOAD = 50_000
if not RUN_50K_DOWNLOAD:
    print('50k SynthHuman pilot download skipped. Set RUN_50K_DOWNLOAD=True when ready; this downloads 10 chunks, not the full 330 GB release.')
else:
    import subprocess, sys, zipfile, json, requests, re, shutil
    from tqdm.auto import tqdm
    from pathlib import Path
    if not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/AppleCapstoneRelighting')
    SYNTHHUMAN_PARENT = DRIVE_ROOT / 'dataset' / 'sources'
    SYNTHHUMAN_ROOT = SYNTHHUMAN_PARENT / 'SynthHuman'
    DOWNLOAD_STATE = SYNTHHUMAN_PARENT / '.synthhuman_download_state'
    DOWNLOAD_CACHE = SYNTHHUMAN_PARENT / '.synthhuman_zip_cache'
    DAVID_REPO = Path('/content/DAViD')
    for folder in (DOWNLOAD_STATE, DOWNLOAD_CACHE): folder.mkdir(parents=True,exist_ok=True)
    if not (DAVID_REPO / 'download_data.py').is_file():
        subprocess.run(['git','clone','--depth','1','https://github.com/microsoft/DAViD.git',str(DAVID_REPO)],check=True)
    TOTAL_SAMPLES, CHUNK_SIZE = 298008, 5000
    SAMPLE_COUNT_TO_DOWNLOAD = min(SAMPLE_COUNT_TO_DOWNLOAD, TOTAL_SAMPLES)
    CHUNKS_TO_DOWNLOAD = (SAMPLE_COUNT_TO_DOWNLOAD + CHUNK_SIZE - 1) // CHUNK_SIZE
    print(f'Will download {SAMPLE_COUNT_TO_DOWNLOAD:,} samples from the first {CHUNKS_TO_DOWNLOAD} chunks.')
    BASE_URL = 'https://facesyntheticspubwedata.z6.web.core.windows.net/iccv-2025/data'
    for chunk_idx in range(CHUNKS_TO_DOWNLOAD):
        first_id=chunk_idx*CHUNK_SIZE
        sample_count=min(CHUNK_SIZE,TOTAL_SAMPLES-first_id)
        last_id=first_id+sample_count-1
        marker=DOWNLOAD_STATE/f'SynthHuman_{chunk_idx:04d}.complete'
        zip_path=DOWNLOAD_CACHE/f'SynthHuman_{chunk_idx:04d}.zip'
        # A legacy completion marker may only mean the ZIP was extracted and deleted.
        # Skip only when the archive itself is present; otherwise reacquire this ZIP.
        if marker.exists() and zip_path.is_file():
            print(f'Chunk {chunk_idx+1:02d}/60 archive already present; skipping download.')
            continue
        if marker.exists() and not zip_path.exists():
            print(f'Chunk {chunk_idx+1:02d}/60 has an old completion marker but no ZIP; downloading the missing archive.')
        url=f'{BASE_URL}/SynthHuman_{chunk_idx:04d}.zip'
        print(f'Downloading chunk {chunk_idx+1:02d}/60 ({sample_count} samples)...')
        # Reuse a complete ZIP if the runtime stopped during extraction; otherwise resume its .part file.
        partial_path=zip_path.with_suffix(zip_path.suffix+'.part')
        if zip_path.exists():
            print(f'Archive already downloaded; reusing {zip_path.name} without downloading it again.')
        else:
            offset=partial_path.stat().st_size if partial_path.exists() else 0
            headers={'Range':f'bytes={offset}-'} if offset else {}
            with requests.get(url,headers=headers,stream=True,timeout=(30,180)) as response:
                response.raise_for_status()
                # If the server ignores Range, restart cleanly instead of appending duplicate bytes.
                if offset and response.status_code != 206:
                    offset=0
                content_length=int(response.headers.get('Content-Length',0)) or None
                total=(offset+content_length) if content_length else None
                print(f'Chunk {chunk_idx+1:02d}/60 download:')
                with open(partial_path,'ab' if offset else 'wb') as output, tqdm(
                    total=total, initial=offset, desc=f'SynthHuman_{chunk_idx:04d}.zip',
                    unit='B', unit_scale=True, unit_divisor=1024, mininterval=1
                ) as progress:
                    for block in response.iter_content(chunk_size=8*1024*1024):
                        if block:
                            output.write(block)
                            progress.update(len(block))
                    output.flush()
            partial_path.replace(zip_path)
        expected_names={f'{kind}_{sample_id:07d}.{ext}' for sample_id in range(first_id,last_id+1) for kind,ext in (('rgb','png'),('alpha','png'),('depth','exr'),('normal','exr'))}
        with zipfile.ZipFile(zip_path) as archive:
            member_by_name={}
            for info in archive.infolist():
                if info.is_dir(): continue
                name=Path(info.filename).name
                if name in expected_names:
                    if name in member_by_name: raise RuntimeError(f'Duplicate file in chunk ZIP: {name}')
                    member_by_name[name]=info
            missing=sorted(expected_names-set(member_by_name))
            if missing: raise RuntimeError(f'Chunk {chunk_idx:04d} ZIP is missing {len(missing)} expected files; first missing: {missing[:10]}')
            # Read every expected member to EOF: zipfile checks each member CRC. Keep the ZIP on Drive.
            with tqdm(total=len(expected_names),desc=f'Verify chunk {chunk_idx+1:02d}/60',unit='file',mininterval=1) as progress:
                for name in sorted(expected_names):
                    bytes_read=0
                    with archive.open(member_by_name[name]) as member:
                        while True:
                            block=member.read(8*1024*1024)
                            if not block: break
                            bytes_read+=len(block)
                    if bytes_read==0: raise RuntimeError(f'Empty file in ZIP: {name}')
                    progress.update(1)
        marker.write_text(json.dumps({'chunk':chunk_idx,'first_sample_id':first_id,'last_sample_id':last_id,'sample_count':sample_count,'verified_file_count':len(expected_names),'zip_crc_verified':True,'archive':str(zip_path),'url':url},indent=2))
        print(f'Chunk {chunk_idx+1:02d}/60 complete; ZIP retained on Drive and all {len(expected_names)} expected members passed CRC verification.')
    print(f'50k pilot archive pass finished ({SAMPLE_COUNT_TO_DOWNLOAD:,} requested samples). ZIPs retained at:',DOWNLOAD_CACHE)

## Audit inventaris unduhan (tanpa memindai folder file gambar)
Cell ini menghitung chunk selesai dari marker dan memeriksa jumlah anggota RGB/alpha/depth/normal pada ZIP yang tersimpan. Marker lama dari mode ekstraksi ditandai sebagai legacy; jumlahnya merupakan catatan proses sebelumnya, bukan audit ulang file fisik.


In [ ]:
# Per-chunk inventory: count RGB images, each label type, complete samples, and totals.
# This reads ZIP central directories or extracted file names; it does not decode image pixels.
import json, os, re, zipfile
from pathlib import Path
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_ROOT = Path('/content/drive/MyDrive/AppleCapstoneRelighting')
SOURCE_ROOT = DRIVE_ROOT / 'dataset' / 'sources'
EXTRACTED_ROOT = SOURCE_ROOT / 'SynthHuman'
ZIP_ROOT = SOURCE_ROOT / '.synthhuman_zip_cache'
STATE_ROOT = SOURCE_ROOT / '.synthhuman_download_state'
TOTAL_SAMPLES, CHUNK_SIZE = 298008, 5000
TOTAL_CHUNKS = (TOTAL_SAMPLES + CHUNK_SIZE - 1) // CHUNK_SIZE
PATTERN = re.compile(r'^(rgb|alpha|depth|normal)_(\d+)\.(png|exr)$')
MODALITIES = ('rgb', 'alpha', 'depth', 'normal')
rows = []

for chunk_idx in range(TOTAL_CHUNKS):
    first_id = chunk_idx * CHUNK_SIZE
    expected_samples = min(CHUNK_SIZE, TOTAL_SAMPLES - first_id)
    last_id = first_id + expected_samples - 1
    counts = {kind: set() for kind in MODALITIES}
    source_kind, status, detail = 'missing', 'MISSING', ''
    zip_path = ZIP_ROOT / f'SynthHuman_{chunk_idx:04d}.zip'
    marker_path = STATE_ROOT / f'SynthHuman_{chunk_idx:04d}.complete'

    if zip_path.is_file():
        source_kind = 'ZIP'
        try:
            with zipfile.ZipFile(zip_path) as archive:
                names = (Path(info.filename).name for info in archive.infolist() if not info.is_dir())
                for name in names:
                    match = PATTERN.match(name)
                    if match:
                        kind, raw_id, _ext = match.groups()
                        sample_id = int(raw_id)
                        if first_id <= sample_id <= last_id:
                            counts[kind].add(sample_id)
            status = 'OK' if all(len(counts[k]) == expected_samples for k in MODALITIES) else 'INCOMPLETE'
        except Exception as exc:
            status, detail = 'ZIP_READ_ERROR', str(exc)
    else:
        chunk_dirs = [EXTRACTED_ROOT / f'chunk_{chunk_idx:04d}', EXTRACTED_ROOT / f'chunk_{chunk_idx:04d}_buckets']
        chunk_dir = next((path for path in chunk_dirs if path.is_dir()), None)
        if chunk_dir is not None:
            source_kind = 'EXTRACTED'
            try:
                # Inspect names only. Bucket subdirectories are recursively enumerated; files aren't stat'ed.
                pending_dirs = [chunk_dir]
                while pending_dirs:
                    current_dir = pending_dirs.pop()
                    with os.scandir(current_dir) as entries:
                        for entry in entries:
                            if entry.name.startswith('bucket_'):
                                try:
                                    if entry.is_dir(follow_symlinks=False): pending_dirs.append(Path(entry.path))
                                except OSError:
                                    pass
                            match = PATTERN.match(entry.name)
                            if match:
                                kind, raw_id, _ext = match.groups()
                                sample_id = int(raw_id)
                                if first_id <= sample_id <= last_id:
                                    counts[kind].add(sample_id)
                status = 'OK' if all(len(counts[k]) == expected_samples for k in MODALITIES) else 'INCOMPLETE'
            except OSError as exc:
                status, detail = 'DRIVE_IO_ERROR', str(exc)
        elif marker_path.is_file():
            source_kind, status = 'MARKER_ONLY', 'NOT_AUDITED'
            detail = 'completion marker exists, but ZIP and recognized extracted chunk folder are unavailable'

    modality_counts = {kind: len(counts[kind]) for kind in MODALITIES}
    complete_ids = set.intersection(*(counts[kind] for kind in MODALITIES)) if all(counts.values()) else set()
    complete_samples = len(complete_ids)
    missing_files = sum(expected_samples - modality_counts[k] for k in MODALITIES) if source_kind in ('ZIP','EXTRACTED') and status != 'DRIVE_IO_ERROR' and status != 'ZIP_READ_ERROR' else None
    rows.append({'chunk':chunk_idx+1,'expected':expected_samples,'complete':complete_samples,'counts':modality_counts,'missing_files':missing_files,'source':source_kind,'status':status,'detail':detail})

print('chunk | complete samples / expected | RGB | alpha | depth | normal | missing modality files | source/status')
print('-' * 112)
for row in rows:
    missing_text = str(row['missing_files']) if row['missing_files'] is not None else 'unknown'
    print(f"{row['chunk']:02d} | {row['complete']:4d}/{row['expected']:<4d} | {row['counts']['rgb']:5d} | {row['counts']['alpha']:5d} | {row['counts']['depth']:5d} | {row['counts']['normal']:6d} | {missing_text:>6} | {row['source']}/{row['status']}")
    if row['detail']: print('   ', row['detail'])

actual_totals = {kind: sum(row['counts'][kind] for row in rows) for kind in MODALITIES}
known_complete_total = sum(row['complete'] for row in rows)
marked_chunks = {idx for idx in range(TOTAL_CHUNKS) if (STATE_ROOT / f'SynthHuman_{idx:04d}.complete').is_file()}
marked_samples = sum(min(CHUNK_SIZE, TOTAL_SAMPLES - idx*CHUNK_SIZE) for idx in marked_chunks)
unknown_chunks = [row['chunk'] for row in rows if row['status'] in ('NOT_AUDITED','DRIVE_IO_ERROR','ZIP_READ_ERROR')]
print('-' * 112)
print(f"Samples inventoried from accessible ZIPs/extracted folders: {known_complete_total:,} / {TOTAL_SAMPLES:,}")
print(f"Samples covered by completion markers (includes legacy markers): {marked_samples:,} / {TOTAL_SAMPLES:,}")
print(f"Modality files inventoried: RGB={actual_totals['rgb']:,}, alpha={actual_totals['alpha']:,}, depth={actual_totals['depth']:,}, normal={actual_totals['normal']:,}")
print(f"RGB images inventoried: {actual_totals['rgb']:,}; expected RGB images: {TOTAL_SAMPLES:,}")
print(f"Total modality files expected for full release: {TOTAL_SAMPLES*4:,}")
if unknown_chunks: print('Chunks with marker but not physically audited (1-based):', unknown_chunks)


## Queue semua ZIP yang tersedia ke disk lokal Colab
Cell ini mengantrekan semua ZIP SynthHuman yang ada di Drive dan mengekstraknya berurutan ke `/content/SynthHuman_staged/chunk_XXXX`. ZIP sumber tetap di Drive. Sebelum mulai, cell menghitung total ukuran hasil ekstrak dan menghentikan proses jika ruang lokal Colab tidak cukup. File staging lokal akan hilang saat runtime dihapus.


In [ ]:
# Queue and stage all available ZIP chunks sequentially on Colab local disk.
import json, re, shutil, zipfile
from pathlib import Path
from tqdm.auto import tqdm
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_ROOT = Path('/content/drive/MyDrive/AppleCapstoneRelighting')
ZIP_ROOT = DRIVE_ROOT / 'dataset' / 'sources' / '.synthhuman_zip_cache'
LOCAL_ROOT = Path('/content/SynthHuman_staged')
TOTAL_SAMPLES, CHUNK_SIZE = 298008, 5000
MAX_SAMPLES_TO_STAGE = 50_000
RESERVE_BYTES = 3 * 1024**3
PATTERN = re.compile(r'^(rgb|alpha|depth|normal)_(\d+)\.(png|exr)$')
MODALITIES = (('rgb','png'),('alpha','png'),('depth','exr'),('normal','exr'))

zip_paths = sorted(ZIP_ROOT.glob('SynthHuman_*.zip'))
if not zip_paths: raise FileNotFoundError(f'No completed ZIP archives found under {ZIP_ROOT}')
queue=[]
required_total=0
for zip_path in zip_paths:
    match=re.fullmatch(r'SynthHuman_(\d{4})\.zip',zip_path.name)
    if not match: continue
    chunk_idx=int(match.group(1))
    first_id=chunk_idx*CHUNK_SIZE
    sample_count=min(CHUNK_SIZE,TOTAL_SAMPLES-first_id)
    last_id=first_id+sample_count-1
    with zipfile.ZipFile(zip_path) as archive:
        expected={f'{kind}_{sample_id:07d}.{ext}' for sample_id in range(first_id,last_id+1) for kind,ext in MODALITIES}
        member_by_name={}
        for info in archive.infolist():
            if info.is_dir(): continue
            name=Path(info.filename).name
            if name in expected: member_by_name[name]=info
        missing=expected-set(member_by_name)
        if missing: raise RuntimeError(f'{zip_path.name} is missing {len(missing)} expected entries; examples: {sorted(missing)[:8]}')
        expanded_bytes=sum(member_by_name[name].file_size for name in expected)
    queue.append((chunk_idx,zip_path,first_id,last_id,sample_count,expanded_bytes))
    required_total+=expanded_bytes

free_bytes=shutil.disk_usage('/content').free
print(f'Staging cap: first {MAX_SAMPLES_TO_STAGE:,} samples. ZIP chunks queued: {len(queue)}; samples represented: {sum(item[4] for item in queue):,}')
print(f'Estimated extracted size: {required_total/1024**3:.1f} GiB; local free: {free_bytes/1024**3:.1f} GiB; reserve: {RESERVE_BYTES/1024**3:.1f} GiB')
if free_bytes < required_total + RESERVE_BYTES:
    raise OSError('Local Colab disk is too small for the whole ZIP queue. No extraction was started. Keep archives on Drive and stage only the chunks needed for the next training run.')

for queue_pos,(chunk_idx,zip_path,first_id,last_id,sample_count,expanded_bytes) in enumerate(queue,1):
    chunk_dir=LOCAL_ROOT/f'chunk_{chunk_idx:04d}'
    chunk_dir.mkdir(parents=True,exist_ok=True)
    expected={f'{kind}_{sample_id:07d}.{ext}' for sample_id in range(first_id,last_id+1) for kind,ext in MODALITIES}
    with zipfile.ZipFile(zip_path) as archive:
        member_by_name={Path(info.filename).name:info for info in archive.infolist() if not info.is_dir() and Path(info.filename).name in expected}
        with tqdm(total=len(expected),desc=f'Queue {queue_pos}/{len(queue)} · chunk {chunk_idx+1:02d}/60',unit='file',mininterval=1) as progress:
            for name in sorted(expected):
                target=chunk_dir/name
                bytes_written=0
                with archive.open(member_by_name[name]) as source,target.open('wb') as output:
                    while True:
                        block=source.read(8*1024*1024)
                        if not block: break
                        output.write(block); bytes_written+=len(block)
                if bytes_written==0: raise RuntimeError(f'Empty extracted file: {name} in {zip_path.name}')
                progress.update(1)
    print(f'Chunk {chunk_idx+1:02d}/60 staged: {sample_count:,} samples at {chunk_dir}')
print(f'ZIP queue complete. Local staging root: {LOCAL_ROOT}. Source archives remain on Drive: {ZIP_ROOT}')

In [ ]:
# Delete only ZIPs whose complete expected contents are present in local staging.
import re, zipfile
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AppleCapstoneRelighting')
ZIP_ROOT = DRIVE_ROOT / 'dataset' / 'sources' / '.synthhuman_zip_cache'
LOCAL_ROOT = Path('/content/SynthHuman_staged')
TOTAL_SAMPLES, CHUNK_SIZE = 298008, 5000
MODALITIES = (('rgb','png'),('alpha','png'),('depth','exr'),('normal','exr'))

zip_paths = sorted(ZIP_ROOT.glob('SynthHuman_*.zip'))
verified, skipped = [], []
for zip_path in zip_paths:
    match = re.fullmatch(r'SynthHuman_(\d{4})\.zip', zip_path.name)
    if not match:
        skipped.append((zip_path.name, 'nama ZIP tidak dikenali')); continue
    chunk_idx = int(match.group(1)); first_id = chunk_idx * CHUNK_SIZE
    if first_id >= TOTAL_SAMPLES:
        skipped.append((zip_path.name, 'di luar rentang dataset')); continue
    last_id = min(first_id + CHUNK_SIZE, TOTAL_SAMPLES)
    chunk_dir = LOCAL_ROOT / f'chunk_{chunk_idx:04d}'
    missing = []
    try:
        with zipfile.ZipFile(zip_path) as archive:
            by_name = {Path(i.filename).name: i for i in archive.infolist() if not i.is_dir()}
            for sample_id in range(first_id, last_id):
                for kind, ext in MODALITIES:
                    name = f'{kind}_{sample_id:07d}.{ext}'
                    info = by_name.get(name); target = chunk_dir / name
                    if info is None or not target.is_file() or target.stat().st_size != info.file_size:
                        missing.append(name)
                        if len(missing) >= 10: break
                if len(missing) >= 10: break
        if missing:
            skipped.append((zip_path.name, f'hasil ekstrak belum lengkap/cocok; contoh: {missing[:3]}'))
        else:
            verified.append(zip_path)
    except Exception as exc:
        skipped.append((zip_path.name, f'gagal diverifikasi: {exc}'))

print(f'ZIP aman dihapus setelah verifikasi staging: {len(verified)}')
for path in verified: print('  ', path.name)
print(f'ZIP dipertahankan: {len(skipped)}')
for name, reason in skipped: print(f'  KEEP {name}: {reason}')
if verified:
    confirmation = input('Ketik DELETE VERIFIED ZIPS untuk menghapus hanya ZIP yang lolos verifikasi di atas: ')
    if confirmation == 'DELETE VERIFIED ZIPS':
        for path in verified:
            path.unlink()
            print(f'Dihapus: {path.name}')
        print('Selesai. ZIP yang tidak lolos verifikasi tetap tersimpan di Drive.')
    else:
        print('Dibatalkan; tidak ada ZIP yang dihapus.')
else:
    print('Tidak ada ZIP yang dihapus.')
